# Inputs for Neural Net
- Type of game played
- Length of game (time and moves)
- Game outcomes
    - Winner
    - Type of outcome
- Move evaluation
    - Variance
    - Mean
    - IQR
    - Next best moves
        - Mean
        - Did they choose the best move?
        - Did they choose the next best move?
- Piece evaluation
    - Number of moves of each piece
    - Captures per piece
    - Number of checks
    - Checks per piece
    - Breakdown per part of the game

# Environment Notes
To get the project to run properly, the python version needs to be set to 3.11. Create a fresh conda envioronment. Run the following commands:
```bash
conda create -n chess python-3.11
conda activate chess
pip install pandas
pip install matplotlib
pip install seaborn
pip install chess
pip install tensorflow
```

In [266]:
# Import Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import chess

# Loading and Cleaning

In [250]:
# Import data and observe shape and columns
chess_data_raw = pd.read_csv("data/games.csv")
print(chess_data_raw.shape)
print(chess_data_raw.columns)
chess_data_raw.head()

chess_data_raw = chess_data_raw[chess_data_raw['turns'] >= 4]

chess_data = chess_data_raw.drop(columns=["id", "white_id", "black_id"])
chess_data = chess_data.dropna()

# Drop all non-rated games for doing rating analysis
chess_data = chess_data[chess_data["rated"] == True]
# Remove rated, created at, and last move at columns
chess_data = chess_data.drop(["rated","created_at","last_move_at"],axis=1)
# Map winner column
chess_data["winner"] = chess_data["winner"].map({"black": -1, "white": 1}).fillna(0)
# Map victory column
chess_data["victory_status"] = chess_data["victory_status"].map({"resign": 0, "mate": 1,"draw": 2, "outoftime": 3})
# Convert moves column to array of moves
chess_data["moves"] = chess_data["moves"].str.split(" ")
# Add column for average player ELO per game
chess_data["game_rating"] = (chess_data["white_rating"] + chess_data["black_rating"]) / 2
# Add column to bucket games by rating 
chess_data["rating_bucket"] = pd.cut(
    chess_data["game_rating"],
    bins=3,
    labels=["low", "medium", "high"]
)

print(chess_data.shape)
chess_data.head()

(20058, 16)
Index(['id', 'rated', 'created_at', 'last_move_at', 'turns', 'victory_status',
       'winner', 'increment_code', 'white_id', 'white_rating', 'black_id',
       'black_rating', 'moves', 'opening_eco', 'opening_name', 'opening_ply'],
      dtype='str')
(15943, 12)


,turns,victory_status,winner,increment_code,white_rating,black_rating,moves,opening_eco,opening_name,opening_ply,game_rating,rating_bucket
1,16,0,-1.0,5+10,1322,1261,"[d4, Nc6, e4, e5, f4, f6, dxe5, fxe5, fxe5, Nx...",B00,Nimzowitsch Defense: Kennedy Variation,4,1291.5,low
2,61,1,1.0,5+10,1496,1500,"[e4, e5, d3, d6, Be3, c6, Be2, b5, Nd2, a5, a4...",C20,King's Pawn Game: Leonardis Variation,3,1498.0,medium
3,61,1,1.0,20+0,1439,1454,"[d4, d5, Nf3, Bf5, Nc3, Nf6, Bf4, Ng4, e3, Nc6...",D02,Queen's Pawn Game: Zukertort Variation,3,1446.5,medium
4,95,1,1.0,30+3,1523,1469,"[e4, e5, Nf3, d6, d4, Nc6, d5, Nb4, a3, Na6, N...",C41,Philidor Defense,5,1496.0,medium
6,33,0,1.0,10+0,1520,1423,"[d4, d5, e4, dxe4, Nc3, Nf6, f3, exf3, Nxf3, N...",D00,Blackmar-Diemer Gambit: Pietrowsky Defense,10,1471.5,medium


In [251]:
# -------------------------------------------------------
# Use Python Chess to get parameters for neural net
# -------------------------------------------------------


def board_analysis(movelist):
    piece_lookup = {
        chess.PAWN: "pawn",
        chess.KNIGHT: "knight",
        chess.BISHOP: "bishop",
        chess.ROOK: "rook",
        chess.QUEEN: "queen",
        chess.KING: "king",
    }
    piece_values = {
        chess.PAWN: 1,
        chess.KNIGHT: 3,
        chess.BISHOP: 3,
        chess.ROOK: 5,
        chess.QUEEN: 9,
        chess.KING: 0,
    }
    colors = ["white", "black"]
    categories = [
        "moves",
        "captures",
        "checks",
        "forks",
        "pins",
        "attacks",
        "defenses",
        "center_control_turns",
    ]
    pieces = ["pawn", "knight", "bishop", "rook", "queen", "king"]
    center_squares = [chess.D4, chess.E4, chess.D5, chess.E5]

    analysis = {
        color: {
            category: {piece: 0 for piece in pieces}
            for category in categories
        }
        for color in colors
    }
    promotion_counts = {color: 0 for color in colors}
    forced_move_counts = {color: 0 for color in colors}
    material_taken = {color: 0 for color in colors}
    material_lost = {color: 0 for color in colors}

    board = chess.Board()

    for san in movelist:
        try:
            move = board.parse_san(san)
        except Exception:
            break

        color = "white" if board.turn == chess.WHITE else "black"
        opponent_color = "black" if color == "white" else "white"
        piece = board.piece_at(move.from_square)

        if piece is None:
            break

        piece_name = piece_lookup[piece.piece_type]
        analysis[color]["moves"][piece_name] += 1

        captured_value = 0
        if board.is_capture(move):
            analysis[color]["captures"][piece_name] += 1
            if board.is_en_passant(move):
                captured_value = piece_values[chess.PAWN]
            else:
                captured_piece = board.piece_at(move.to_square)
                if captured_piece is not None:
                    captured_value = piece_values[captured_piece.piece_type]

        material_taken[color] += captured_value
        material_lost[opponent_color] += captured_value

        board.push(move)

        if board.is_check():
            analysis[color]["checks"][piece_name] += 1

        attacked_enemy_pieces = 0
        defended_friendly_pieces = 0
        pinned_enemy_pieces = 0
        for square in board.attacks(move.to_square):
            target_piece = board.piece_at(square)
            if target_piece is None:
                continue
            if target_piece.color != piece.color:
                attacked_enemy_pieces += 1
                if board.is_pinned(target_piece.color, square):
                    pinned_enemy_pieces += 1
            else:
                defended_friendly_pieces += 1

        analysis[color]["attacks"][piece_name] += attacked_enemy_pieces
        analysis[color]["defenses"][piece_name] += defended_friendly_pieces

        if attacked_enemy_pieces >= 2:
            analysis[color]["forks"][piece_name] += 1

        if pinned_enemy_pieces > 0:
            analysis[color]["pins"][piece_name] += pinned_enemy_pieces

        if move.promotion is not None:
            promotion_counts[color] += 1

        if board.legal_moves.count() == 1:
            forced_move_counts[color] += 1

        for square in center_squares:
            center_piece = board.piece_at(square)
            if center_piece is None:
                continue
            center_color = "white" if center_piece.color == chess.WHITE else "black"
            center_piece_name = piece_lookup[center_piece.piece_type]
            analysis[center_color]["center_control_turns"][center_piece_name] += 1

    flattened_vector = []
    for color in colors:
        for category in categories:
            for piece in pieces:
                flattened_vector.append(analysis[color][category][piece])

    for color in colors:
        flattened_vector.extend([
            promotion_counts[color],
            forced_move_counts[color],
            material_taken[color],
            material_lost[color],
        ])

    return flattened_vector


In [252]:
colors = ["white", "black"]
categories = [
    "moves",
    "captures",
    "checks",
    "forks",
    "pins",
    "attacks",
    "defenses",
    "center_control_turns",
]
pieces = ["pawn", "knight", "bishop", "rook", "queen", "king"]
summary_metrics = ["promotions", "forced_moves", "material_taken", "material_lost"]

neural_net_columns = [
    f"{color}_{category}_{piece}"
    for color in colors
    for category in categories
    for piece in pieces
] + [
    f"{color}_{metric}"
    for color in colors
    for metric in summary_metrics
]

neural_net_columns += [
    "white_elo",
    "black_elo",
    "winner",
    "total_turns",
    "white_turns",
    "black_turns",
    "idx"
]

neural_net_rows = []

for idx, row in chess_data.iterrows():
    total_turns = row["turns"]
    white_turns = (total_turns + 1) // 2
    black_turns = total_turns // 2

    board_analysis_row = board_analysis(row["moves"])
    board_analysis_row += [
        row["white_rating"],
        row["black_rating"],
        row["winner"],
        total_turns,
        white_turns,
        black_turns,
        idx
    ]
    neural_net_rows.append(board_analysis_row)

neural_net_input = pd.DataFrame(neural_net_rows, columns=neural_net_columns)
neural_net_input.describe()


,white_moves_pawn,white_moves_knight,white_moves_bishop,white_moves_rook,white_moves_queen,white_moves_king,white_captures_pawn,white_captures_knight,white_captures_bishop,white_captures_rook,...,black_forced_moves,black_material_taken,black_material_lost,white_elo,black_elo,winner,total_turns,white_turns,black_turns,idx
count,15943.000000,15943.00000,15943.000000,15943.000000,15943.000000,15943.000000,15943.000000,15943.000000,15943.000000,15943.000000,...,15943.000000,15943.000000,15943.000000,15943.00000,15943.000000,15943.000000,15943.000000,15943.000000,15943.000000,15943.000000
mean,8.607288,5.57831,4.966067,4.442828,4.010977,4.032052,1.841247,1.363984,1.430095,1.172803,...,0.353008,20.199272,20.374020,1599.90949,1596.395660,0.041147,62.756758,31.637521,31.119237,10121.965502
std,4.381908,3.43769,3.048665,5.252713,3.749370,5.677335,1.375202,1.241930,1.195479,1.490381,...,0.908748,11.310281,11.261918,282.71278,287.500823,0.976612,33.240935,16.615902,16.628786,5789.764497
min,1.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,784.00000,789.000000,-1.000000,4.000000,2.000000,2.000000,1.000000
25%,5.000000,3.00000,3.000000,1.000000,2.000000,1.000000,1.000000,0.000000,1.000000,0.000000,...,0.000000,11.000000,11.000000,1399.00000,1393.000000,-1.000000,39.000000,20.000000,19.000000,5159.500000
50%,8.000000,5.00000,4.000000,3.000000,3.000000,2.000000,2.000000,1.000000,1.000000,1.000000,...,0.000000,20.000000,20.000000,1579.00000,1576.000000,0.000000,57.000000,29.000000,28.000000,10047.000000
75%,11.000000,7.00000,6.000000,6.000000,5.000000,5.000000,3.000000,2.000000,2.000000,2.000000,...,0.000000,29.000000,30.000000,1795.00000,1793.000000,1.000000,81.000000,41.000000,40.000000,15223.500000
max,35.000000,59.00000,44.000000,83.000000,41.000000,74.000000,8.000000,11.000000,11.000000,9.000000,...,16.000000,55.000000,55.000000,2622.00000,2588.000000,1.000000,349.000000,175.000000,174.000000,20057.000000


In [253]:
neural_net_input

,white_moves_pawn,white_moves_knight,white_moves_bishop,white_moves_rook,white_moves_queen,white_moves_king,white_captures_pawn,white_captures_knight,white_captures_bishop,white_captures_rook,...,black_forced_moves,black_material_taken,black_material_lost,white_elo,black_elo,winner,total_turns,white_turns,black_turns,idx
0,6,0,0,0,2,0,2,0,0,0,...,0,11,2,1322,1261,-1.0,16,8,8,1
1,12,3,8,5,3,0,7,1,3,3,...,0,7,38,1496,1500,1.0,61,31,30,2
2,4,9,4,3,10,1,1,3,2,0,...,0,8,30,1439,1454,1.0,61,31,30,3
3,13,8,3,16,4,4,1,3,1,5,...,1,30,38,1523,1469,1.0,95,48,47,4
4,3,4,5,0,4,1,0,2,0,0,...,0,9,9,1520,1423,1.0,33,17,16,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15938,3,4,3,1,0,1,0,1,1,0,...,0,4,2,1691,1220,1.0,24,12,12,20053
15939,8,11,3,2,8,9,2,3,0,0,...,1,26,25,1233,1196,-1.0,82,41,41,20054
15940,3,6,5,0,4,0,0,0,2,0,...,0,6,4,1219,1286,1.0,35,18,17,20055
15941,24,10,4,9,2,6,5,3,2,2,...,2,31,38,1360,1227,1.0,109,55,54,20056


# Nueral Network Development and Design
- Experiment with multiple types of models
- Use smaller data to test
- Find which performs the best
- Train the best performing model using large data

## Import Tensorflow and Relevant Libraries
- Using tensorflow because it is flexible and fast
- It will allow us to process the relatively large dataset (13k rows)

In [254]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Strategy Statistics Model
- Inputs:
    - Action counters (checks, forks, pins, attacks, defenses, moves, and center-control turns per piece per color)
    - Game-level counters (promotions, forced moves, and material taken/lost per color)
    - No centipawn data

- Question: Can ELO be predicted without stockfish centipawn calculation and purely based off of board state and piece statistics?

In [255]:
# ==============================
# Seperate data into X and y
# ==============================

neural_net_input["game_elo"] = neural_net_input[["white_elo", "black_elo"]].mean(axis=1)

feature_cols = [col for col in neural_net_input.columns if col not in ["white_elo", "black_elo", "game_elo"]]
target_cols = ["game_elo"]

X = neural_net_input[feature_cols].values.astype(np.float32)
y = neural_net_input[target_cols].values.astype(np.float32)

# ============================================================
# Split data into test, train, and validation
# ============================================================

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=42
)

# ===================================
# Create normalization layer (scaling)
# ===================================

normalizer = layers.Normalization(axis=-1)
normalizer.adapt(X_train)

# ==============================
# Build model
# ==============================
model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    normalizer,
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])


model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=[keras.metrics.MeanAbsoluteError()]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    )
]

# ==============================
# Train
# ==============================
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# ==============================
# Evaluate
# ==============================
pred = model.predict(X_test, verbose=0)

game_mae = mean_absolute_error(y_test[:, 0], pred[:, 0])
game_rmse = np.sqrt(mean_squared_error(y_test[:, 0], pred[:, 0]))

print(f"Game ELO MAE:  {game_mae:.2f}")
print(f"Game ELO RMSE: {game_rmse:.2f}")

Epoch 1/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 666us/step - loss: 882519.0000 - mean_absolute_error: 747.3882 - val_loss: 163420.5000 - val_mean_absolute_error: 319.8758
Epoch 2/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 475us/step - loss: 116911.1562 - mean_absolute_error: 268.1515 - val_loss: 110995.3828 - val_mean_absolute_error: 260.4337
Epoch 3/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 477us/step - loss: 90242.7344 - mean_absolute_error: 235.7446 - val_loss: 95095.2109 - val_mean_absolute_error: 241.5401
Epoch 4/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 475us/step - loss: 79243.3828 - mean_absolute_error: 221.6550 - val_loss: 86418.3906 - val_mean_absolute_error: 230.4464
Epoch 5/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 474us/step - loss: 72405.1016 - mean_absolute_error: 212.5496 - val_loss: 80490.6094 - val_mean_absolute_error: 222.6328
Epoch 6/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 472us/step - loss: 67476.6250 - mean_absolute_error: 205.7114 - val_loss: 76155.8438 - val_mean_absolute_error: 216.9381
Epoch 

# Analysis
This model performed quite poorly. The mean error was over 200, which is not ideal for ELO. ELO ranges from around 100 - ~3400. The middle 50% of players in this dataset are within 1400 - 1800. This means that our error is too large to accurately guess ELO.

Conclusion: ELO cannot be accurately guessed purely by the strategy statistics calculated for the model. ...

# Neural Net Using Buckets Instead of Regression
- Buckets for classification are high, medium, and low ELO
- Low: <1400
- Medium: 1400-1800
- High: >1800

In [256]:
# ==============================
# Create logical rating buckets
# ==============================
from sklearn.metrics import accuracy_score, classification_report

neural_net_input["average_elo"] = neural_net_input[["white_elo", "black_elo"]].mean(axis=1)
bucket_bins = [0, 1400, 1800, np.inf]
bucket_labels = ["low", "medium", "high"]
neural_net_input["rating_bucket"] = pd.cut(
    neural_net_input["average_elo"],
    bins=bucket_bins,
    labels=bucket_labels,
    include_lowest=True,
    right=False
)

# ==============================
# Prepare bucket classification data
# ==============================
bucket_feature_cols = [
    col for col in neural_net_input.columns
    if col not in ["white_elo", "black_elo", "average_elo", "rating_bucket"]
]
bucket_to_int = {label: idx for idx, label in enumerate(bucket_labels)}

X_bucket = neural_net_input[bucket_feature_cols].values.astype(np.float32)
y_bucket = neural_net_input["rating_bucket"].map(bucket_to_int).astype(np.int32).values

X_bucket_train_full, X_bucket_test, y_bucket_train_full, y_bucket_test = train_test_split(
    X_bucket, y_bucket, test_size=0.15, random_state=42, stratify=y_bucket
)

X_bucket_train, X_bucket_val, y_bucket_train, y_bucket_val = train_test_split(
    X_bucket_train_full,
    y_bucket_train_full,
    test_size=0.15,
    random_state=42,
    stratify=y_bucket_train_full
)

# ===================================
# Create normalization layer (scaling)
# ===================================
bucket_normalizer = layers.Normalization(axis=-1)
bucket_normalizer.adapt(X_bucket_train)

# ==============================
# Build bucket classifier
# ==============================
bucket_model = keras.Sequential([
    keras.Input(shape=(X_bucket_train.shape[1],)),
    bucket_normalizer,
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(len(bucket_labels), activation='softmax')
])

bucket_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

bucket_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    )
]

bucket_history = bucket_model.fit(
    X_bucket_train, y_bucket_train,
    validation_data=(X_bucket_val, y_bucket_val),
    epochs=200,
    batch_size=32,
    callbacks=bucket_callbacks,
    verbose=1
)

bucket_pred_probs = bucket_model.predict(X_bucket_test, verbose=0)
bucket_pred = np.argmax(bucket_pred_probs, axis=1)
bucket_accuracy = accuracy_score(y_bucket_test, bucket_pred)

print(f"Bucket accuracy: {bucket_accuracy:.3f}")
print(classification_report(y_bucket_test, bucket_pred, target_names=bucket_labels))


Epoch 1/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 702us/step - accuracy: 0.7581 - loss: 0.5408 - val_accuracy: 0.9046 - val_loss: 0.2175
Epoch 2/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 480us/step - accuracy: 0.9226 - loss: 0.1867 - val_accuracy: 0.9257 - val_loss: 0.1700
Epoch 3/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 478us/step - accuracy: 0.9528 - loss: 0.1162 - val_accuracy: 0.9410 - val_loss: 0.1563
Epoch 4/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 482us/step - accuracy: 0.9677 - loss: 0.0851 - val_accuracy: 0.9336 - val_loss: 0.1645
Epoch 5/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 477us/step - accuracy: 0.9714 - loss: 0.0787 - val_accuracy: 0.9415 - val_loss: 0.1669
Epoch 6/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 478us/step - accuracy: 0.9743 - loss: 0.0645 - val_accuracy: 0.9297 - val_loss: 0.1991
Epoch 7/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 486us/step - accuracy: 0.9812 - loss: 0.0498 - val_accuracy: 0.9326 - val_loss: 0.2134
Epoch 8/200
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 480us/step - accuracy: 0.9832 - loss: 0

# Centipawn Model
- Data processing
- Neural network

In [257]:
import ast

# ==============================
# Data processing
# ==============================

centipawn_data = pd.read_csv("centipawn/centipawn_data_depth_12.csv")

# Clamp values (stockfish provided very high values 100000 and -100000 when checkmate was soon)
centipawn_data["centipawn"] = centipawn_data["centipawn"].apply(ast.literal_eval)
centipawn_data["centipawn"] = centipawn_data["centipawn"].apply(
    lambda values: [max(-1000, min(1000, value)) for value in values]
)

# Sort centipawn by color
centipawn_data["white_centipawn"] = centipawn_data["centipawn"].apply(lambda x: x[::2])
centipawn_data["black_centipawn"] = centipawn_data["centipawn"].apply(lambda x: x[1::2])

# Combine centipawn data with board-analysis features using the explicit index columns
neural_net_with_centipawn = neural_net_input.merge(
    centipawn_data,
    left_on="idx",
    right_on="index",
    how="inner"
)

centipawn_vector_input = neural_net_with_centipawn[neural_net_with_centipawn["total_turns"] >= 40]

centipawn_vector_input = centipawn_vector_input[["white_centipawn", "black_centipawn", "white_elo", "black_elo"]].copy()
centipawn_vector_input["game_elo"] = centipawn_vector_input[["white_elo", "black_elo"]].mean(axis=1)
centipawn_vector_input

,white_centipawn,black_centipawn,white_elo,black_elo,game_elo
1,"[39, 42, 25, 18, 70, 73, 260, 506, 571, 552, 5...","[-38, 15, -6, -14, -46, -84, -268, -504, -530,...",1496,1500,1498.0
2,"[43, 37, 48, 0, 89, 133, 172, 232, 382, 437, 5...","[-34, -32, 13, -5, -69, -70, -105, -199, -262,...",1439,1454,1446.5
3,"[40, 45, 74, 91, 117, 115, 111, 82, 45, -75, -...","[-45, -43, -72, -88, -118, -115, -82, -43, 55,...",1523,1469,1496.0
5,"[47, 46, 32, 114, 53, 51, 44, 15, 10, 1, -24, ...","[-47, 12, -33, -58, -51, -41, -14, -2, -3, 21,...",1439,1392,1415.5
6,"[36, 76, 82, 89, 180, 175, 59, 159, 184, 106, ...","[-47, -73, -81, -92, -184, -66, -55, -153, -77...",1381,1209,1295.0
...,...,...,...,...,...
15934,"[41, 60, 40, 85, 54, 51, 19, 39, -4, -74, -88,...","[-36, -40, -6, -44, -30, -25, -19, 22, 87, 84,...",1328,1252,1290.0
15936,"[43, 59, 38, 7, -189, 22, -271, -350, -159, -1...","[-38, -35, 13, 254, 181, 277, 324, 442, 184, 3...",1237,1231,1234.0
15939,"[43, 64, 132, 111, 8, 23, 15, 18, 21, -28, 82,...","[-43, -48, 2, -9, -2, -4, -17, -15, 74, 22, 26...",1233,1196,1214.5
15941,"[42, 61, 66, -123, -121, -45, -62, -122, -154,...","[-40, -69, 120, 119, 130, 93, 120, 142, 213, 2...",1360,1227,1293.5


In [258]:
# ==============================
# Flatten first 20 centipawn moves per side
# ==============================
centipawn_vector_input = centipawn_vector_input[
    (centipawn_vector_input["white_centipawn"].apply(len) >= 20) &
    (centipawn_vector_input["black_centipawn"].apply(len) >= 20)
].copy()

white_cp_columns = [f"white_move_{i + 1}" for i in range(20)]
black_cp_columns = [f"black_move_{i + 1}" for i in range(20)]

white_cp_df = pd.DataFrame(
    centipawn_vector_input["white_centipawn"].apply(lambda moves: moves[:20]).tolist(),
    columns=white_cp_columns,
    index=centipawn_vector_input.index
)

black_cp_df = pd.DataFrame(
    centipawn_vector_input["black_centipawn"].apply(lambda moves: moves[:20]).tolist(),
    columns=black_cp_columns,
    index=centipawn_vector_input.index
)

centipawn_flat_input = pd.concat(
    [
        white_cp_df,
        black_cp_df,
        centipawn_vector_input[["game_elo"]]
    ],
    axis=1
)

centipawn_flat_input.head()

# ==============================
# Prepare data for dense neural net
# ==============================
cp_feature_cols = white_cp_columns + black_cp_columns
cp_target_cols = ["game_elo"]

X_cp = centipawn_flat_input[cp_feature_cols].values.astype(np.float32)
y_cp = centipawn_flat_input[cp_target_cols].values.astype(np.float32)

X_cp_train_full, X_cp_test, y_cp_train_full, y_cp_test = train_test_split(
    X_cp, y_cp, test_size=0.15, random_state=42
)

X_cp_train, X_cp_val, y_cp_train, y_cp_val = train_test_split(
    X_cp_train_full, y_cp_train_full, test_size=0.15, random_state=42
)

# ===================================
# Create normalization layer (scaling)
# ===================================
cp_normalizer = layers.Normalization(axis=-1)
cp_normalizer.adapt(X_cp_train)

# ==============================
# Build dense centipawn model
# ==============================
cp_model = keras.Sequential([
    keras.Input(shape=(40,)),
    cp_normalizer,
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.1),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(1)
])

cp_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss="mse",
    metrics=[keras.metrics.MeanAbsoluteError()]
)

cp_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=30,
        restore_best_weights=True
    )
]

cp_history = cp_model.fit(
    X_cp_train, y_cp_train,
    validation_data=(X_cp_val, y_cp_val),
    epochs=200,
    batch_size=32,
    callbacks=cp_callbacks,
    verbose=1
)

# ==============================
# Evaluate
# ==============================
cp_pred = cp_model.predict(X_cp_test, verbose=0)

cp_game_mae = mean_absolute_error(y_cp_test[:, 0], cp_pred[:, 0])
cp_game_rmse = np.sqrt(mean_squared_error(y_cp_test[:, 0], cp_pred[:, 0]))

print(f"Game ELO MAE:  {cp_game_mae:.2f}")
print(f"Game ELO RMSE: {cp_game_rmse:.2f}")


Epoch 1/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 1s 885us/step - loss: 1828853.7500 - mean_absolute_error: 1249.8488 - val_loss: 1067928.7500 - val_mean_absolute_error: 895.3959
Epoch 2/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 628us/step - loss: 777290.7500 - mean_absolute_error: 747.5193 - val_loss: 468286.1562 - val_mean_absolute_error: 564.4819
Epoch 3/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 787us/step - loss: 282107.9062 - mean_absolute_error: 422.0841 - val_loss: 165290.4844 - val_mean_absolute_error: 312.4846
Epoch 4/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 624us/step - loss: 130710.7578 - mean_absolute_error: 280.5849 - val_loss: 104583.8828 - val_mean_absolute_error: 249.8507
Epoch 5/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 631us/step - loss: 94829.8203 - mean_absolute_error: 240.9922 - val_loss: 83313.5781 - val_mean_absolute_error: 224.9286
Epoch 6/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 625us/step - loss: 77302.8047 - mean_absolute_error: 219.4095 - val_loss: 70256.8984 - val_mean_absolute_error: 208.6019

# Adding summary statistics to centipawn data

In [259]:
summary_data = neural_net_with_centipawn[neural_net_with_centipawn["total_turns"] >= 40].copy()
summary_data = summary_data[["white_centipawn", "black_centipawn", "white_elo", "black_elo", "index"]]

white_avg = []
white_std = []
white_median = []
white_max = []
white_min = []

black_avg = []
black_std = []
black_median = []
black_max = []
black_min = []

for idx, row in summary_data.iterrows():
    white_cp = row["white_centipawn"]
    white_avg.append(np.mean(white_cp))
    white_std.append(np.std(white_cp))
    white_median.append(np.median(white_cp))
    white_max.append(np.max(white_cp))
    white_min.append(np.min(white_cp))

    black_cp = row["black_centipawn"]
    black_avg.append(np.mean(black_cp))
    black_std.append(np.std(black_cp))
    black_median.append(np.median(black_cp))
    black_max.append(np.max(black_cp))
    black_min.append(np.min(black_cp))

summary_data["white_avg"] = white_avg
summary_data["white_std"] = white_std
summary_data["white_median"] = white_median
summary_data["white_max"] = white_max
summary_data["white_min"] = white_min

summary_data["black_avg"] = black_avg
summary_data["black_std"] = black_std
summary_data["black_median"] = black_median
summary_data["black_max"] = black_max
summary_data["black_min"] = black_min


summary_input = pd.concat(
    [
        white_cp_df,
        black_cp_df,
        summary_data[["white_avg", "white_std", "white_median", "white_max", "white_min", "black_avg", "black_std", "black_median", "black_max", "black_min"]]
    ],
    axis=1
)

summary_input["game_elo"] = (summary_data["white_elo"] + summary_data["black_elo"]) / 2

In [267]:
# ==============================
# Prepare summary data for dense neural net
# ==============================
summary_feature_cols = [col for col in summary_input.columns if col not in ["game_elo"]]
summary_target_cols = ["game_elo"]

X_summary = summary_input[summary_feature_cols].values.astype(np.float32)
y_summary = summary_input[summary_target_cols].values.astype(np.float32)

X_summary_train_full, X_summary_test, y_summary_train_full, y_summary_test = train_test_split(
    X_summary, y_summary, test_size=0.15, random_state=42
)

X_summary_train, X_summary_val, y_summary_train, y_summary_val = train_test_split(
    X_summary_train_full, y_summary_train_full, test_size=0.15, random_state=42
)

# ===================================
# Create normalization layer (scaling)
# ===================================
summary_normalizer = layers.Normalization(axis=-1)
summary_normalizer.adapt(X_summary_train)

# ==============================
# Build combined model
# ==============================
summary_model = keras.Sequential([
    keras.Input(shape=(X_summary_train.shape[1],)),
    summary_normalizer,
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)
])

summary_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=[keras.metrics.MeanAbsoluteError()]
)

summary_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True
    )
]

summary_history = summary_model.fit(
    X_summary_train, y_summary_train,
    validation_data=(X_summary_val, y_summary_val),
    epochs=200,
    batch_size=32,
    callbacks=summary_callbacks,
    verbose=1
)

# ==============================
# Evaluate
# ==============================
summary_pred = summary_model.predict(X_summary_test, verbose=0)

summary_game_mae = mean_absolute_error(y_summary_test[:, 0], summary_pred[:, 0])
summary_game_rmse = np.sqrt(mean_squared_error(y_summary_test[:, 0], summary_pred[:, 0]))

print(f"Game ELO MAE:  {summary_game_mae:.2f}")
print(f"Game ELO RMSE: {summary_game_rmse:.2f}")

# Save best-performing model
os.makedirs("weights", exist_ok=True)
summary_model.save("weights/flattened_centipawn_summary.keras")


Epoch 1/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 1s 722us/step - loss: 1415256.3750 - mean_absolute_error: 1052.2020 - val_loss: 400750.8125 - val_mean_absolute_error: 519.8005
Epoch 2/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 486us/step - loss: 184400.1406 - mean_absolute_error: 332.3985 - val_loss: 89594.9531 - val_mean_absolute_error: 234.9057
Epoch 3/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 496us/step - loss: 92086.3203 - mean_absolute_error: 241.1132 - val_loss: 70652.8828 - val_mean_absolute_error: 211.3830
Epoch 4/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 482us/step - loss: 79212.1797 - mean_absolute_error: 223.3206 - val_loss: 63537.4844 - val_mean_absolute_error: 201.4609
Epoch 5/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 484us/step - loss: 71549.0000 - mean_absolute_error: 213.4410 - val_loss: 58598.4727 - val_mean_absolute_error: 194.2038
Epoch 6/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 478us/step - loss: 65404.6953 - mean_absolute_error: 203.8358 - val_loss: 54274.0273 - val_mean_absolute_error: 186.4379
Epoch

# Combining the methods
- Take both strategy statistics and game evaluation to provide more accurate ELO prediction

In [261]:
combined_neural_net_input = pd.concat(
    [
        white_cp_df,
        black_cp_df,
        neural_net_with_centipawn[neural_net_with_centipawn["total_turns"] >= 40]
    ],
    axis=1
)

combined_neural_net_input["game_elo"] = combined_neural_net_input[["white_elo", "black_elo"]].mean(axis=1)
combined_neural_net_input = combined_neural_net_input.drop(columns=["centipawn", "black_centipawn", "white_centipawn", "idx", "index", "rating_bucket", "total_turns", "white_turns", "black_turns", "average_elo"])
combined_neural_net_input

,white_move_1,white_move_2,white_move_3,white_move_4,white_move_5,white_move_6,white_move_7,white_move_8,white_move_9,white_move_10,...,white_material_taken,white_material_lost,black_promotions,black_forced_moves,black_material_taken,black_material_lost,white_elo,black_elo,winner,game_elo
1,39,42,25,18,70,73,260,506,571,552,...,38,7,0,0,7,38,1496,1500,1.0,1498.0
2,43,37,48,0,89,133,172,232,382,437,...,30,8,0,0,8,30,1439,1454,1.0,1446.5
3,40,45,74,91,117,115,111,82,45,-75,...,38,30,0,1,30,38,1523,1469,1.0,1496.0
5,47,46,32,114,53,51,44,15,10,1,...,12,24,0,3,24,12,1439,1392,-1.0,1415.5
6,36,76,82,89,180,175,59,159,184,106,...,38,37,0,0,37,38,1381,1209,1.0,1295.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15934,41,60,40,85,54,51,19,39,-4,-74,...,15,20,0,0,20,15,1328,1252,1.0,1290.0
15936,43,59,38,7,-189,22,-271,-350,-159,-181,...,10,15,0,1,15,10,1237,1231,-1.0,1234.0
15939,43,64,132,111,8,23,15,18,21,-28,...,25,26,0,1,26,25,1233,1196,-1.0,1214.5
15941,42,61,66,-123,-121,-45,-62,-122,-154,-199,...,38,31,0,2,31,38,1360,1227,1.0,1293.5


In [262]:
# ==============================
# Prepare combined data for dense neural net
# ==============================
combined_feature_cols = [col for col in combined_neural_net_input.columns if col not in ["white_elo", "black_elo", "game_elo"]]
combined_target_cols = ["game_elo"]

X_combined = combined_neural_net_input[combined_feature_cols].values.astype(np.float32)
y_combined = combined_neural_net_input[combined_target_cols].values.astype(np.float32)

X_combined_train_full, X_combined_test, y_combined_train_full, y_combined_test = train_test_split(
    X_combined, y_combined, test_size=0.15, random_state=42
)

X_combined_train, X_combined_val, y_combined_train, y_combined_val = train_test_split(
    X_combined_train_full, y_combined_train_full, test_size=0.15, random_state=42
)

# ===================================
# Create normalization layer (scaling)
# ===================================
combined_normalizer = layers.Normalization(axis=-1)
combined_normalizer.adapt(X_combined_train)

# ==============================
# Build combined model
# ==============================
combined_model = keras.Sequential([
    keras.Input(shape=(X_combined_train.shape[1],)),
    combined_normalizer,
    layers.Dense(256, activation="relu"),
    # layers.Dropout(0.2),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(1)
])

combined_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss="mse",
    metrics=[keras.metrics.MeanAbsoluteError()]
)

combined_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=30,
        restore_best_weights=True
    )
]

combined_history = combined_model.fit(
    X_combined_train, y_combined_train,
    validation_data=(X_combined_val, y_combined_val),
    epochs=200,
    batch_size=32,
    callbacks=combined_callbacks,
    verbose=1
)

# ==============================
# Evaluate
# ==============================
combined_pred = combined_model.predict(X_combined_test, verbose=0)

combined_game_mae = mean_absolute_error(y_combined_test[:, 0], combined_pred[:, 0])
combined_game_rmse = np.sqrt(mean_squared_error(y_combined_test[:, 0], combined_pred[:, 0]))

print(f"Game ELO MAE:  {combined_game_mae:.2f}")
print(f"Game ELO RMSE: {combined_game_rmse:.2f}")


Epoch 1/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 1s 879us/step - loss: 1464557.7500 - mean_absolute_error: 1065.3337 - val_loss: 314714.4062 - val_mean_absolute_error: 448.7460
Epoch 2/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 671us/step - loss: 202874.0312 - mean_absolute_error: 357.0383 - val_loss: 152840.8594 - val_mean_absolute_error: 302.8763
Epoch 3/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 683us/step - loss: 123110.1328 - mean_absolute_error: 277.1586 - val_loss: 117953.6797 - val_mean_absolute_error: 264.6151
Epoch 4/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 682us/step - loss: 99436.3594 - mean_absolute_error: 249.4286 - val_loss: 101532.3750 - val_mean_absolute_error: 245.3784
Epoch 5/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 671us/step - loss: 86443.1406 - mean_absolute_error: 233.1993 - val_loss: 92093.7656 - val_mean_absolute_error: 233.8998
Epoch 6/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 683us/step - loss: 77648.2969 - mean_absolute_error: 221.6138 - val_loss: 85715.2969 - val_mean_absolute_error: 225.8239
E

In [263]:
# ==============================
# Baseline comparison: predict mean game ELO
# ==============================
baseline_mean_pred = np.full((len(y_combined_test), 1), y_combined_train_full.mean())

baseline_game_mae = mean_absolute_error(y_combined_test[:, 0], baseline_mean_pred[:, 0])
baseline_game_rmse = np.sqrt(mean_squared_error(y_combined_test[:, 0], baseline_mean_pred[:, 0]))

comparison_df = pd.DataFrame(
    {
        "model": ["Mean baseline", "Combined neural net"],
        "game_mae": [baseline_game_mae, combined_game_mae],
        "game_rmse": [baseline_game_rmse, combined_game_rmse],
    }
)

print("Improvement over mean baseline (positive is better):")
print(f"Game ELO MAE improvement:  {baseline_game_mae - combined_game_mae:.2f}")
print(f"Game ELO RMSE improvement: {baseline_game_rmse - combined_game_rmse:.2f}")

comparison_df


Improvement over mean baseline (positive is better):
Game ELO MAE improvement:  21.09
Game ELO RMSE improvement: 20.45


,model,game_mae,game_rmse
0,Mean baseline,207.597778,256.346979
1,Combined neural net,186.511200,235.895750


# Strategy Board Statistics + Centipawn Summary Statistics Model

In [264]:
# ==============================
# Combine board strategy stats with centipawn summary stats
# ==============================
board_strategy_cols = [
    col for col in neural_net_input.columns
    if col not in ["white_elo", "black_elo", "game_elo", "winner", "total_turns", "white_turns", "black_turns", "idx", "average_elo", "rating_bucket"]
]

centipawn_summary_cols = [
    "white_avg", "white_std", "white_median", "white_max", "white_min",
    "black_avg", "black_std", "black_median", "black_max", "black_min"
]

strategy_summary_input = pd.concat(
    [
        neural_net_with_centipawn.loc[summary_data.index, board_strategy_cols].reset_index(drop=True),
        summary_data[centipawn_summary_cols].reset_index(drop=True)
    ],
    axis=1
)

strategy_summary_input["game_elo"] = (
    summary_data["white_elo"].reset_index(drop=True) +
    summary_data["black_elo"].reset_index(drop=True)
) / 2

# ==============================
# Prepare data for dense neural net
# ==============================
strategy_summary_feature_cols = [col for col in strategy_summary_input.columns if col != "game_elo"]
strategy_summary_target_cols = ["game_elo"]

X_strategy_summary = strategy_summary_input[strategy_summary_feature_cols].values.astype(np.float32)
y_strategy_summary = strategy_summary_input[strategy_summary_target_cols].values.astype(np.float32)

X_strategy_summary_train_full, X_strategy_summary_test, y_strategy_summary_train_full, y_strategy_summary_test = train_test_split(
    X_strategy_summary, y_strategy_summary, test_size=0.15, random_state=42
)

X_strategy_summary_train, X_strategy_summary_val, y_strategy_summary_train, y_strategy_summary_val = train_test_split(
    X_strategy_summary_train_full, y_strategy_summary_train_full, test_size=0.15, random_state=42
)

# ===================================
# Create normalization layer (scaling)
# ===================================
strategy_summary_normalizer = layers.Normalization(axis=-1)
strategy_summary_normalizer.adapt(X_strategy_summary_train)

# ==============================
# Build model
# ==============================
strategy_summary_model = keras.Sequential([
    keras.Input(shape=(X_strategy_summary_train.shape[1],)),
    strategy_summary_normalizer,
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(1)
])

strategy_summary_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=[keras.metrics.MeanAbsoluteError()]
)

strategy_summary_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True
    )
]

strategy_summary_history = strategy_summary_model.fit(
    X_strategy_summary_train, y_strategy_summary_train,
    validation_data=(X_strategy_summary_val, y_strategy_summary_val),
    epochs=200,
    batch_size=32,
    callbacks=strategy_summary_callbacks,
    verbose=1
)

# ==============================
# Evaluate
# ==============================
strategy_summary_pred = strategy_summary_model.predict(X_strategy_summary_test, verbose=0)

strategy_summary_mae = mean_absolute_error(y_strategy_summary_test[:, 0], strategy_summary_pred[:, 0])
strategy_summary_rmse = np.sqrt(mean_squared_error(y_strategy_summary_test[:, 0], strategy_summary_pred[:, 0]))

print(f"Game ELO MAE:  {strategy_summary_mae:.2f}")
print(f"Game ELO RMSE: {strategy_summary_rmse:.2f}")


Epoch 1/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 1s 897us/step - loss: 674653.8750 - mean_absolute_error: 603.6559 - val_loss: 116429.9688 - val_mean_absolute_error: 263.6765
Epoch 2/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 673us/step - loss: 100306.4922 - mean_absolute_error: 249.8323 - val_loss: 91377.7578 - val_mean_absolute_error: 234.1760
Epoch 3/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 662us/step - loss: 83032.9688 - mean_absolute_error: 228.7443 - val_loss: 80817.3281 - val_mean_absolute_error: 220.8622
Epoch 4/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 665us/step - loss: 73339.5469 - mean_absolute_error: 215.6395 - val_loss: 74727.5078 - val_mean_absolute_error: 212.5519
Epoch 5/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 670us/step - loss: 66062.0000 - mean_absolute_error: 204.9029 - val_loss: 71468.1250 - val_mean_absolute_error: 208.3394
Epoch 6/200
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 781us/step - loss: 62140.2266 - mean_absolute_error: 198.8329 - val_loss: 68449.5938 - val_mean_absolute_error: 204.5295
Epoch 7